# BylawCheck-Lite: experiments

Compares one small instruct model family (Qwen2.5) on a building-regulation Q&A task over three PDFs (Delhi, Mumbai, Hyderabad):

| # | Variant | Retrieval | Weights |
|---|---------|-----------|---------|
| 1 | Base, no RAG | no | fp16 |
| 2 | Base + RAG | yes | fp16 |
| 3 | LoRA, no RAG | no | fp16 + LoRA |
| 4 | LoRA + RAG | yes | fp16 + LoRA |
| 5 | QLoRA, no RAG | no | 4-bit + LoRA |
| 6 | QLoRA + RAG | yes | 4-bit + LoRA |
| 7 | Extractive baseline (best sentence of the top chunk, no LLM) | yes | none |

Variant 7 is added so we can see whether the LLM beats a trivial "return the retrieved text" system.

**Run on Kaggle:** GPU T4 x2 or P100, internet on, add the three PDFs as a dataset (`delhi.pdf`, `mumbai.pdf`, `hyderabad.pdf`). Only the first GPU is used. Set `QUICK_RUN = False` below for the full run (about 3 to 5 hours on one T4).

In [ ]:
QUICK_RUN = True            # True: tiny data, 1 epoch (smoke test). False: full run.

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # model under test (try Qwen2.5-3B-Instruct too)
GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"      # writes the Q&A data and acts as judge
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
N_TEST_ANSWERABLE = 12 if QUICK_RUN else 45
N_TEST_UNANSWERABLE = 6 if QUICK_RUN else 15
N_TRAIN_QA = 40 if QUICK_RUN else 500
EPOCHS = 1 if QUICK_RUN else 3
LORA_R = 16
TOP_K = 5
ABSTAIN_THRESHOLD = 0.65    # same constant as pipeline.py in the app
USE_JUDGE = not QUICK_RUN   # optional LLM-as-judge score (1-5)
MAX_NEW_TOKENS = 128 if QUICK_RUN else 200

## 1. Setup and installs

Versions are pinned. **If the install fails or Kaggle's preinstalled packages clash** (usual suspects: `bitsandbytes`, `numpy`, `langchain-*`), restart the session after the install cell and run again; if `bitsandbytes` still fails, the QLoRA rows are skipped automatically and the rest of the notebook still runs.

In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 bitsandbytes==0.44.1 datasets==3.1.0 \
    sentence-transformers==3.3.1 faiss-cpu==1.9.0.post1 pymupdf==1.24.14 \
    langchain==0.3.27 langchain-core==0.3.76 langchain-community==0.3.27 langchain-text-splitters==0.3.11 \
    langchain-huggingface==0.3.1 langgraph==0.6.7 matplotlib pandas

In [ ]:
import os, re, glob, json, random, time, gc, math, warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # one GPU only: avoids DataParallel trouble with 4-bit models
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from transformers import set_seed

random.seed(SEED); np.random.seed(SEED); set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this notebook needs a GPU)")

def find_pdf(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"data/{name}")
    assert hits, f"{name} not found: add it as a Kaggle dataset"
    return hits[0]

PDFS = {"Delhi": find_pdf("delhi.pdf"), "Mumbai": find_pdf("mumbai.pdf"), "Hyderabad": find_pdf("hyderabad.pdf")}
print(PDFS)

## 2. PDF extraction

Same logic as `build_index.py` in the app: one document per PDF page, page numbers are 1-based PDF indexes. A PDF whose average text per page is tiny has no text layer and would need OCR.

In [ ]:
import fitz  # PyMuPDF
from langchain_core.documents import Document

def load_pages(city, path):
    pdf = fitz.open(path)
    texts = [p.get_text() for p in pdf]
    docs = [Document(page_content=t, metadata={"city": city, "page": i + 1}) for i, t in enumerate(texts) if t.strip()]
    return docs, len(texts)

pages, n_pdf_pages = {}, {}
for city, path in PDFS.items():
    pages[city], n_pdf_pages[city] = load_pages(city, path)

stats = pd.DataFrame({
    "pdf_pages": n_pdf_pages,
    "pages_with_text": {c: len(p) for c, p in pages.items()},
    "chars": {c: sum(len(d.page_content) for d in p) for c, p in pages.items()},
})
stats["chars_per_page"] = (stats.chars / stats.pdf_pages).round(0)
stats["no_text_layer"] = stats.chars_per_page < 50
print(stats)

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
stats.pdf_pages.plot.bar(ax=ax[0], title="Pages per city", color="#1F4E3D", rot=0)
(stats.chars / 1e6).plot.bar(ax=ax[1], title="Characters per city (millions)", color="#1F4E3D", rot=0)
plt.tight_layout(); plt.show()
if stats.no_text_layer.any():
    print("WARNING: no text layer in", list(stats.index[stats.no_text_layer]), "- OCR needed (see build_index.py)")

## 3. Chunking, embeddings, FAISS

Identical to the app: `RecursiveCharacterTextSplitter` (2000 chars, about 500 tokens, overlap 320), bge-small embeddings (normalised), FAISS with `city` and `page` metadata. Chunks never cross page boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=320)
chunks = []
for city in PDFS:
    chunks += [c for c in splitter.split_documents(pages[city]) if len(c.page_content.strip()) > 80]
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cuda"},
                            encode_kwargs={"normalize_embeddings": True})
store = FAISS.from_documents(chunks, emb)

def retrieve(query, city="All", k=TOP_K):
    """(Document, L2 distance) pairs; smaller distance = more similar."""
    flt = None if city == "All" else {"city": city}
    return store.similarity_search_with_score(query, k=k, filter=flt, fetch_k=store.index.ntotal)

chunk_df = pd.DataFrame({"city": [c.metadata["city"] for c in chunks], "chars": [len(c.page_content) for c in chunks]})
print(chunk_df.city.value_counts())
chunk_df.chars.plot.hist(bins=40, color="#1F4E3D", title="Chunk length (characters)", figsize=(6, 3)); plt.show()

## 4. RAG pipeline as a LangGraph graph

Same structure as `ask_graph` in the app: `retrieve -> check_relevance -> (generate_answer | abstain) -> END`. The only difference is the LLM, here the local Qwen model instead of Groq.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

REFUSAL = "I can't find this in the provided regulations."
SYS_RAG = ("You answer questions about building regulations for Delhi, Mumbai and Hyderabad. "
           "Answer ONLY from the provided context. Cite every claim as [City, p.X] using the labels in the context. "
           "Never invent numbers: if a number is not in the context, do not state it. "
           f"If the context does not contain the answer, reply exactly: {REFUSAL} Be concise.")
SYS_PLAIN = ("You are an expert on Indian building regulations (Delhi, Mumbai, Hyderabad). Answer concisely from your own "
             f"knowledge. If you do not know, reply exactly: {REFUSAL}")

def format_context(docs):
    return "\n\n".join(f"[{d.metadata['city']}, p.{d.metadata['page']}]\n{d.page_content}" for d in docs)

def rag_messages(q, docs):
    return [{"role": "system", "content": SYS_RAG}, {"role": "user", "content": f"Context:\n{format_context(docs)}\n\nQuestion: {q}"}]

def plain_messages(q):
    return [{"role": "system", "content": SYS_PLAIN}, {"role": "user", "content": f"Question: {q}"}]

def is_refusal(text):
    return "can't find this in the provided regulations" in text.replace("’", "'").lower()

def load_tok(name=MODEL_NAME, side="left"):
    return AutoTokenizer.from_pretrained(name, padding_side=side)

def load_model(kind="fp16", name=MODEL_NAME, adapter=None):
    """kind: 'fp16' or '4bit'. The adapter (LoRA/QLoRA) is attached on top when given."""
    kw = dict(device_map={"": 0}, torch_dtype=torch.float16)
    if kind == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(name, **kw)
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
    return model.eval()

def clear_gpu():
    gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def generate(model, tok, messages_list, max_new_tokens=MAX_NEW_TOKENS, batch_size=4):
    """Greedy decoding for a list of chat-message lists."""
    outs = []
    for i in range(0, len(messages_list), batch_size):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from IPython.display import Image, display

class AskState(TypedDict, total=False):
    question: str
    city: str
    hits: list
    relevant: bool
    answer: str
    abstained: bool

def make_ask_graph(model, tok):
    def retrieve_node(s):
        return {"hits": retrieve(s["question"], s.get("city", "All"), TOP_K)}
    def check_relevance(s):
        best = min((sc for _, sc in s["hits"]), default=float("inf"))
        return {"relevant": best <= ABSTAIN_THRESHOLD}
    def generate_answer(s):
        ans = generate(model, tok, [rag_messages(s["question"], [d for d, _ in s["hits"]])])[0]
        return {"answer": REFUSAL if is_refusal(ans) else ans, "abstained": is_refusal(ans)}
    def abstain(s):
        return {"answer": REFUSAL, "abstained": True}

    g = StateGraph(AskState)
    for name, fn in [("retrieve", retrieve_node), ("check_relevance", check_relevance),
                     ("generate_answer", generate_answer), ("abstain", abstain)]:
        g.add_node(name, fn)
    g.set_entry_point("retrieve")
    g.add_edge("retrieve", "check_relevance")
    g.add_conditional_edges("check_relevance", lambda s: "generate_answer" if s["relevant"] else "abstain")
    g.add_edge("generate_answer", END); g.add_edge("abstain", END)
    return g.compile()

tok = load_tok()
model = load_model("fp16")
ask_graph = make_ask_graph(model, tok)

print(ask_graph.get_graph().draw_mermaid())
print("nodes:", list(ask_graph.get_graph().nodes))
print("edges:", [(e.source, e.target) for e in ask_graph.get_graph().edges])
try:
    display(Image(ask_graph.get_graph().draw_mermaid_png()))   # needs internet (mermaid.ink)
except Exception as e:
    print("diagram image skipped:", type(e).__name__)

In [ ]:
examples = ["What is the minimum front setback for a residential plot in Mumbai?",
            "Is rainwater harvesting required for buildings in Delhi?",
            "What is the best biryani restaurant in Hyderabad?"]
for q in examples:
    out = ask_graph.invoke({"question": q, "city": "All"})
    print("=" * 100, "\nQ:", q)
    for d, sc in out["hits"]:
        print(f"  [{d.metadata['city']}, p.{d.metadata['page']}] dist={sc:.3f}  {' '.join(d.page_content.split())[:110]}...")
    print("ANSWER:", out["answer"], "\nABSTAINED:", out["abstained"])
model = None; ask_graph = None; clear_gpu()

## 5. Evaluation data and fine-tuning data

- **Test set:** about 45 answerable Q&A pairs written by an LLM (`GEN_MODEL`) from random chunks, stratified over the three cities, plus about 15 hand-written unanswerable questions (topics not in the PDFs, or other cities).
- **Fine-tuning set:** about 500 Q&A pairs generated the same way from **other** chunks. Test chunks and their direct overlapping neighbours are removed from the pool first, so no test passage appears in training. A tenth of the examples are refusal examples (question plus an unrelated chunk, answer = the refusal sentence). About half of the answerable examples are formatted with context and half without, so one adapter serves both "no RAG" and "+ RAG".

In [ ]:
rng = random.Random(SEED)

def good(c):  # long enough and rule-like (contains numbers)
    return len(c.page_content) > 800 and sum(ch.isdigit() for ch in c.page_content) >= 10

cand = [c for c in chunks if good(c)]
per_city = N_TEST_ANSWERABLE // 3
test_chunks = []
for city in PDFS:
    pool = [c for c in cand if c.metadata["city"] == city]
    test_chunks += rng.sample(pool, per_city)
test_ids = {c.metadata["chunk_id"] for c in test_chunks}

# remove neighbours that overlap a test chunk on the same page
banned = set(test_ids)
for i in test_ids:
    for j in (i - 1, i + 1):
        if 0 <= j < len(chunks) and chunks[j].metadata["page"] == chunks[i].metadata["page"] \
                and chunks[j].metadata["city"] == chunks[i].metadata["city"]:
            banned.add(j)
train_pool = [c for c in cand if c.metadata["chunk_id"] not in banned]
train_chunks = rng.sample(train_pool, min(N_TRAIN_QA, len(train_pool)))
assert not ({c.metadata["chunk_id"] for c in train_chunks} & banned)
print(f"test chunks: {len(test_chunks)}, train chunks: {len(train_chunks)}, leakage: 0 by construction")

UNANSWERABLE = [
    "What is the GST rate on cement for construction in India?",
    "How do I renew my driving licence?",
    "What are the building bye-laws for Bengaluru?",
    "What is the stamp duty on registering a flat in Pune?",
    "What is the minimum wage for construction workers in Delhi?",
    "How is property tax calculated in Chennai?",
    "What documents are needed for a food business licence?",
    "What are the fire safety rules for ships and offshore platforms?",
    "What is the penalty for tax evasion under the Income Tax Act?",
    "What are the rules for flying drones near airports?",
    "How do I register a trademark in India?",
    "What is the speed limit for vehicles on national highways?",
    "What are the high-rise building rules in Kolkata?",
    "Who designed the Taj Mahal and when was it built?",
    "What is the recipe for a good filter coffee?",
]

In [ ]:
gen_tok = load_tok(GEN_MODEL)
gen_model = load_model("fp16", GEN_MODEL)

def qa_messages(c):
    city, page = c.metadata["city"], c.metadata["page"]
    text = " ".join(c.page_content.split())[:1800]
    return [{"role": "system", "content": "You write exam questions about building regulations."},
            {"role": "user", "content": f"Text from the {city} building regulations (page {page}):\n\n{text}\n\n"
             f"Write ONE specific question that can be answered using only this text, and its short answer "
             f"(1-2 sentences, copy numbers exactly). Mention {city} in the question. Do not say 'the text' or 'the passage'. "
             "Reply in exactly this format:\nQ: <question>\nA: <answer>"}]

def make_qa(chs):
    outs = generate(gen_model, gen_tok, [qa_messages(c) for c in chs], max_new_tokens=120, batch_size=8)
    rows = []
    for c, o in zip(chs, outs):
        if "Q:" in o and "A:" in o:
            q, a = o.split("A:", 1)
            q, a = q.replace("Q:", "").strip(), a.strip()
            if len(q) > 15 and "?" in q and len(a) > 3:
                rows.append({"question": q, "gold": a, "chunk_id": c.metadata["chunk_id"],
                             "city": c.metadata["city"], "page": c.metadata["page"]})
    return rows

test_qa = make_qa(test_chunks)
train_qa = make_qa(train_chunks)
gen_model = None; clear_gpu()
print(len(test_qa), "test Q&A,", len(train_qa), "train Q&A")
pd.DataFrame(test_qa).head(5)

In [ ]:
EVAL = [dict(r, answerable=True) for r in test_qa]
EVAL += [{"question": q, "gold": None, "chunk_id": None, "city": "n/a", "page": None, "answerable": False}
         for q in UNANSWERABLE[:N_TEST_UNANSWERABLE]]
print(len(EVAL), "eval items:", sum(e["answerable"] for e in EVAL), "answerable,", sum(not e["answerable"] for e in EVAL), "unanswerable")

# fine-tuning examples: (messages, target answer)
train_examples = []
for r in train_qa:
    gold_doc = chunks[r["chunk_id"]]
    if rng.random() < 0.5:
        train_examples.append((rag_messages(r["question"], [gold_doc]), f"{r['gold']} [{r['city']}, p.{r['page']}]"))
    else:
        train_examples.append((plain_messages(r["question"]), r["gold"]))
for r in rng.sample(train_qa, max(1, len(train_qa) // 10)):   # refusal examples
    other = rng.choice([c for c in train_pool if c.metadata["page"] != r["page"] or c.metadata["city"] != r["city"]])
    train_examples.append((rag_messages(r["question"], [other]), REFUSAL))
rng.shuffle(train_examples)
n_eval = max(2, len(train_examples) // 10)
ft_eval, ft_train = train_examples[:n_eval], train_examples[n_eval:]
print("fine-tuning examples: train", len(ft_train), "/ eval", len(ft_eval))

## 6. Models to compare

All seven variants from the table at the top, evaluated on the same test set. LoRA trains on the fp16 base, QLoRA on the 4-bit NF4 base (compute dtype fp16, since the T4 has no bf16). Both use r=16 on all attention and MLP projections.

In [ ]:
VARIANTS = ["1 Base, no RAG", "2 Base + RAG", "3 LoRA, no RAG", "4 LoRA + RAG",
            "5 QLoRA, no RAG", "6 QLoRA + RAG", "7 Extractive baseline"]
ADAPTER_DIR = "/kaggle/working/adapters" if os.path.exists("/kaggle/working") else "adapters"
os.makedirs(ADAPTER_DIR, exist_ok=True)

## 7. Training (LoRA and QLoRA)

3 epochs (1 in quick mode), r=16, batch 2 x gradient accumulation 8, lr 2e-4, fp16 mixed precision. Trainable adapter weights are kept in fp32 (required for fp16 AMP). Training loss is the mean of the logged step losses within each epoch; eval loss is measured at the end of each epoch.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset

train_tok = load_tok(side="right")

def encode(msgs, answer, max_len=1024):
    prompt = train_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    p_ids = train_tok(prompt, add_special_tokens=False).input_ids
    a_ids = train_tok(answer + "<|im_end|>", add_special_tokens=False).input_ids
    ids = (p_ids + a_ids)[:max_len]
    labels = ([-100] * len(p_ids) + a_ids)[:max_len]          # loss on the answer only
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

def to_ds(examples):
    rows = [encode(m, a) for m, a in examples]
    return Dataset.from_list([r for r in rows if any(l != -100 for l in r["labels"])])

ds_train, ds_eval = to_ds(ft_train), to_ds(ft_eval)

def epoch_curves(log):
    tr, ev = {}, {}
    for r in log:
        if "loss" in r: tr.setdefault(math.ceil(r["epoch"] - 1e-6), []).append(r["loss"])
        if "eval_loss" in r: ev[round(r["epoch"])] = r["eval_loss"]
    return [float(np.mean(v)) for _, v in sorted(tr.items())], [v for _, v in sorted(ev.items())]

def train_adapter(name, quantized):
    model = load_model("4bit" if quantized else "fp16")
    if quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    cfg = LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()           # fp16 AMP needs fp32 trainable weights
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"[{name}] trainable params: {n_train:,} of {n_total:,} ({100 * n_train / n_total:.2f}%)")

    args = TrainingArguments(
        output_dir=f"ckpt_{name}", num_train_epochs=EPOCHS, per_device_train_batch_size=2, per_device_eval_batch_size=2,
        gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
        fp16=True, logging_steps=1, eval_strategy="epoch", save_strategy="no", report_to="none", seed=SEED,
        gradient_checkpointing=quantized, gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit" if quantized else "adamw_torch", remove_unused_columns=False)
    trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_eval,
                      data_collator=DataCollatorForSeq2Seq(train_tok, padding=True, label_pad_token_id=-100))
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 2 ** 30
    tr, ev = epoch_curves(trainer.state.log_history)
    model.save_pretrained(f"{ADAPTER_DIR}/{name}")
    print(f"[{name}] time {secs / 60:.1f} min, peak GPU memory {peak:.1f} GiB, adapter saved to {ADAPTER_DIR}/{name}")
    del trainer, model; clear_gpu()
    return {"train_loss": tr, "eval_loss": ev, "minutes": secs / 60, "peak_gib": peak, "trainable": n_train}

history = {}
history["lora"] = train_adapter("lora", quantized=False)
try:
    history["qlora"] = train_adapter("qlora", quantized=True)
except Exception as e:   # bitsandbytes problems on some Kaggle images
    print("QLoRA failed, its rows will be skipped:", type(e).__name__, e)

In [ ]:
fig, ax = plt.subplots(1, len(history), figsize=(5 * len(history), 3.5), squeeze=False)
for a, (name, h) in zip(ax[0], history.items()):
    xs = range(1, len(h["train_loss"]) + 1)
    a.plot(xs, h["train_loss"], marker="o", label="train", color="#1F4E3D")
    a.plot(range(1, len(h["eval_loss"]) + 1), h["eval_loss"], marker="s", ls="--", label="eval", color="#B07A00")
    a.set_title(f"{name.upper()} loss"); a.set_xlabel("epoch"); a.set_ylabel("loss"); a.set_xticks(list(xs)); a.legend()
plt.tight_layout(); plt.show()
print(pd.DataFrame(history).T[["trainable", "minutes", "peak_gib"]])

## 8. Evaluation

Every variant answers the same test set. The RAG variants use identical retrieval (top-5, city = All, same abstention threshold), so the differences come from the model only. The loop below is the `ask_graph` logic run in batches for speed: retrieve, abstain if too far, else generate, abstain if the model refuses.

Metrics (answerable questions unless noted):
- **Hit@k**: the gold chunk is among the top-k retrieved.
- **Token F1** against the gold answer (citations removed). A refused answer scores 0.
- **Judge** (optional, 1 to 5, scored by `GEN_MODEL`).
- **Faithfulness proxy**: share of the answer's key terms and numbers that appear in the top-5 retrieved context. For no-RAG variants it is computed against the context that retrieval would have supplied, so it shows how grounded the answer is.
- **Hallucinated numbers**: numbers in the answer that do not appear in that context (mean per answer).
- **Abstention accuracy** over all items: refuses on unanswerable, does not refuse on answerable.

In [ ]:
CITE = re.compile(r"\[[^\]]*p\.\s*\d+[^\]]*\]")
STOP = set("the a an of to in and or for is are be by on at as with that this it from shall may not any under such "
           "which where when than then also will have has been their there these those into upon only".split())

def clean(s): return CITE.sub("", s).replace(",", "").lower()
def toks(s): return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", clean(s))
def nums(s): return re.findall(r"\d+(?:\.\d+)?", clean(s))

def token_f1(pred, gold):
    p, g = toks(pred), toks(gold)
    common = sum(min(p.count(t), g.count(t)) for t in set(p))
    if not p or not g or common == 0: return 0.0
    pr, rc = common / len(p), common / len(g)
    return 2 * pr * rc / (pr + rc)

def faithfulness(ans, ctx):
    terms = [t for t in toks(ans) if t not in STOP and (len(t) > 3 or any(c.isdigit() for c in t))]
    if not terms: return np.nan
    ctx_set = set(toks(ctx))
    return sum(t in ctx_set for t in terms) / len(terms)

def hallucinated(ans, ctx):
    ctx_nums = set(nums(ctx))
    return sum(n not in ctx_nums for n in nums(ans))

# retrieval for every eval item (shared by all RAG variants)
for e in EVAL:
    e["hits"] = retrieve(e["question"], "All", TOP_K)
    e["best"] = min(sc for _, sc in e["hits"])
    e["relevant"] = e["best"] <= ABSTAIN_THRESHOLD
    e["ctx"] = format_context([d for d, _ in e["hits"]])

def hit_at(k, items):
    return float(np.mean([e["chunk_id"] in [d.metadata["chunk_id"] for d, _ in e["hits"][:k]] for e in items]))

ans_items = [e for e in EVAL if e["answerable"]]
hit_rows = {"All": {k: hit_at(k, ans_items) for k in (1, 3, 5)}}
for city in PDFS:
    hit_rows[city] = {k: hit_at(k, [e for e in ans_items if e["city"] == city]) for k in (1, 3, 5)}
hit_df = pd.DataFrame(hit_rows).T.rename(columns=lambda k: f"hit@{k}")
print(hit_df.round(3))
hit_df.plot.bar(figsize=(6, 3), rot=0, title="Retrieval hit@k (gold chunk in top-k)", color=["#C9D8CF", "#7FA690", "#1F4E3D"]); plt.show()

# how well does the distance threshold separate answerable from unanswerable?
plt.figure(figsize=(6, 3))
plt.hist([e["best"] for e in ans_items], bins=15, alpha=.7, label="answerable", color="#1F4E3D")
plt.hist([e["best"] for e in EVAL if not e["answerable"]], bins=15, alpha=.7, label="unanswerable", color="#B07A00")
plt.axvline(ABSTAIN_THRESHOLD, color="k", ls="--", label="ABSTAIN_THRESHOLD"); plt.xlabel("best retrieval distance"); plt.legend(); plt.show()

In [ ]:
def best_sentence(e):
    """Extractive baseline: the sentence of the top chunk sharing most words with the question."""
    top = " ".join(e["hits"][0][0].page_content.split())
    q = set(toks(e["question"])) - STOP
    sents = re.split(r"(?<=[.;:])\s+", top)
    return max(sents, key=lambda s: len(q & set(toks(s))))[:300]

def answer_variant(model, tok, use_rag):
    """Batched version of ask_graph for one model."""
    todo = [e for e in EVAL if (e["relevant"] or not use_rag)]
    msgs = [rag_messages(e["question"], [d for d, _ in e["hits"]]) if use_rag else plain_messages(e["question"]) for e in todo]
    gens = dict(zip(map(id, todo), generate(model, tok, msgs)))
    return [gens.get(id(e), REFUSAL) for e in EVAL]

answers = {}
answers["7 Extractive baseline"] = [best_sentence(e) if e["relevant"] else REFUSAL for e in EVAL]
for kind, label, adapter in [("fp16", "Base", None), ("fp16", "LoRA", f"{ADAPTER_DIR}/lora"), ("4bit", "QLoRA", f"{ADAPTER_DIR}/qlora")]:
    if adapter and not os.path.exists(adapter):
        print("skipping", label, "(no adapter)"); continue
    m = load_model(kind, adapter=adapter)
    n = VARIANTS.index(f"{[1, 3, 5][['Base', 'LoRA', 'QLoRA'].index(label)]} {label}, no RAG")
    answers[VARIANTS[n]] = answer_variant(m, tok, use_rag=False)
    answers[VARIANTS[n + 1]] = answer_variant(m, tok, use_rag=True)
    m = None; clear_gpu()
    print("done:", label)

In [ ]:
JUDGE = {}
if USE_JUDGE:
    jtok, jmodel = load_tok(GEN_MODEL), load_model("fp16", GEN_MODEL)
    def judge_msgs(e, a):
        return [{"role": "system", "content": "You grade answers to building-regulation questions. Score 1-5: 5 = same facts as the "
                 "reference, 3 = partly correct, 1 = wrong, missing, or a refusal. Reply with one digit only."},
                {"role": "user", "content": f"Question: {e['question']}\nReference answer: {e['gold']}\nModel answer: {CITE.sub('', a)}\nScore:"}]
    for v, ans in answers.items():
        outs = generate(jmodel, jtok, [judge_msgs(e, a) for e, a in zip(EVAL, ans) if e["answerable"]], max_new_tokens=3, batch_size=8)
        JUDGE[v] = [int(re.search(r"[1-5]", o).group()) if re.search(r"[1-5]", o) else 1 for o in outs]
    jmodel = None; clear_gpu()

rows = []
for v, ans in answers.items():
    refused = [is_refusal(a) for a in ans]
    for i, (e, a, r) in enumerate(zip(EVAL, ans, refused)):
        rows.append({"variant": v, "q": i, "city": e["city"], "answerable": e["answerable"], "refused": r,
                     "f1": token_f1(a, e["gold"]) if e["answerable"] and not r else 0.0,
                     "faith": faithfulness(a, e["ctx"]) if e["answerable"] and not r else np.nan,
                     "halluc": hallucinated(a, e["ctx"]) if e["answerable"] and not r else np.nan})
df = pd.DataFrame(rows)
if JUDGE:
    df.loc[df.answerable, "judge"] = [s for v in answers for s in JUDGE[v]]

def summarise(g):
    a, u = g[g.answerable], g[~g.answerable]
    out = {"F1": a.f1.mean(), "Faithfulness": a.faith.mean(), "Halluc. numbers/answer": a.halluc.mean(),
           "Abstain acc": ((a.refused == False).sum() + (u.refused == True).sum()) / len(g),
           "Correct refusals": u.refused.mean(), "False refusals": a.refused.mean()}
    if JUDGE: out["Judge (1-5)"] = a.judge.mean()
    return pd.Series(out)

summary = df.groupby("variant").apply(summarise)
summary.insert(0, "Hit@5", [hit_df.loc["All", "hit@5"] if "RAG" in v and "no RAG" not in v or "Extractive" in v else np.nan for v in summary.index])
print("sample answer per variant (first answerable question):", EVAL[0]["question"])
for v, ans in answers.items(): print(f"  {v}: {ans[0][:160]!r}")

## 9. Results

In [ ]:
metrics = [c for c in ["F1", "Faithfulness", "Abstain acc", "Judge (1-5)"] if c in summary.columns]
plot_df = summary[metrics].copy()
if "Judge (1-5)" in plot_df: plot_df["Judge (1-5)"] /= 5; plot_df = plot_df.rename(columns={"Judge (1-5)": "Judge / 5"})
colors = ["#C9D8CF", "#1F4E3D", "#E3D3A8", "#B07A00", "#C7D3E6", "#35597F", "#999999"]
ax = plot_df.T.plot.bar(figsize=(10, 4), rot=0, color=colors[:len(plot_df)], width=.85, title="All variants per metric (higher is better)")
ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

summary["Halluc. numbers/answer"].plot.bar(figsize=(8, 3), color=colors[:len(summary)], rot=20, title="Numbers in the answer not found in the retrieved context (lower is better)")
plt.tight_layout(); plt.show()

fig, axs = plt.subplots(2, 4, figsize=(13, 5.5))
for a, v in zip(axs.flat, df.variant.unique()):
    g = df[df.variant == v]
    cm = [[((~g.answerable) & (~g.refused)).sum(), ((~g.answerable) & g.refused).sum()],
          [(g.answerable & (~g.refused)).sum(), (g.answerable & g.refused).sum()]]
    a.imshow(cm, cmap="Greens"); a.set_title(v, fontsize=8)
    a.set_xticks([0, 1], ["answered", "refused"]); a.set_yticks([0, 1], ["unanswerable", "answerable"])
    for i in range(2):
        for j in range(2): a.text(j, i, cm[i][j], ha="center", va="center")
for a in list(axs.flat)[df.variant.nunique():]: a.axis("off")
plt.suptitle("Abstention confusion matrices"); plt.tight_layout(); plt.show()

city_f1 = df[df.answerable].groupby(["city", "variant"]).f1.mean().unstack()
city_f1.plot.bar(figsize=(10, 3.5), rot=0, color=colors[:city_f1.shape[1]], title="Token F1 per city"); plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

pd.set_option("display.width", 200)
print(summary.round(3))

In [ ]:
from IPython.display import Markdown
f1 = summary["F1"]; fa = summary["Faithfulness"]; ab = summary["Abstain acc"]
best = f1.idxmax()
def delta(a, b, col="F1"):
    return summary.loc[a, col] - summary.loc[b, col] if a in summary.index and b in summary.index else np.nan
lines = [f"**Best token F1:** {best} ({f1.max():.3f}). **Best abstention accuracy:** {ab.idxmax()} ({ab.max():.3f}). "
         f"**Most faithful:** {fa.idxmax()} ({fa.max():.3f}).",
         f"- RAG vs no RAG on the base model: F1 {summary.loc['1 Base, no RAG', 'F1']:.3f} -> {summary.loc['2 Base + RAG', 'F1']:.3f}.",
         f"- LoRA + RAG vs Base + RAG: F1 difference {delta('4 LoRA + RAG', '2 Base + RAG'):+.3f}; QLoRA + RAG vs Base + RAG: {delta('6 QLoRA + RAG', '2 Base + RAG'):+.3f}.",
         f"- Extractive baseline F1: {f1['7 Extractive baseline']:.3f}, versus {f1['2 Base + RAG']:.3f} for Base + RAG.",
         f"- The test set has only {len(ans_items)} answerable and {len(EVAL) - len(ans_items)} unanswerable questions, so differences of a few points are within noise."]
if QUICK_RUN: lines.append("- QUICK_RUN is on (tiny data, 1 epoch): rerun with QUICK_RUN = False before drawing conclusions.")
better = delta("4 LoRA + RAG", "2 Base + RAG")
lines.append("- **Does fine-tuning beat plain RAG?** " + ("Not on these numbers: LoRA + RAG does not beat Base + RAG on F1." if not better > 0
             else "On F1, yes: LoRA + RAG is ahead of Base + RAG (see the difference above and the noise caveat)."))
display(Markdown("### Conclusion (generated from the numbers above)\n\n" + "\n".join(lines)))

## 10. Plan-report demo

Three sample building descriptions, one per city, run through the same flow as the app's `report_graph`: one retrieval query per topic (FSI, height vs road width, setbacks, parking, other), then a single generation. The local base model writes the report; pass `adapter=` to `load_model` to try a fine-tuned one.

In [ ]:
TOPICS = {
    "FSI": "floor space index FSI permissible for {use} building on plot area {plot_area} sq m",
    "height vs road width": "maximum building height permitted with road width {road_width} m for {use} building",
    "setbacks / open space": "front side rear setbacks open space margins for {use} plot {plot_area} sq m height {height} m",
    "parking": "parking requirement number of car parking spaces for {use} building",
    "other": "ground coverage permissible number of storeys {floors} floors {use} building",
}
SYS_REPORT = SYS_RAG + """

Write a short plan pre-check report. Markdown only.
Line 1 exactly: Rating: Looks Good   OR   Rating: Needs Changes   OR   Rating: Not Enough Info
Then '## ' sections in this order: Applicable rules (with [City, p.X]), Looks fine, Needs attention, Could not check.
Anything marked 'not provided', or any rule missing from the context, goes under Could not check. Do not add a disclaimer."""

def report_flow(model, tok, plan):
    fields = {f: plan.get(f, "not provided") for f in ["use", "plot_area", "road_width", "floors", "height"]}
    docs, seen = [], set()
    for q in TOPICS.values():
        for d, _ in retrieve(q.format(**fields), plan["city"], 3):
            if d.metadata["chunk_id"] not in seen:
                seen.add(d.metadata["chunk_id"]); docs.append(d)
    plan_txt = "\n".join(f"- {k}: {v}" for k, v in plan.items())
    msgs = [{"role": "system", "content": SYS_REPORT}, {"role": "user", "content": f"Plan details:\n{plan_txt}\n\nContext:\n{format_context(docs)}"}]
    return generate(model, tok, [msgs], max_new_tokens=350 if QUICK_RUN else 700, batch_size=1)[0]

PLANS = [
    {"city": "Delhi", "use": "Residential", "plot_area": 300, "road_width": 12, "floors": 4, "height": 15, "parking_spaces": 3, "fsi": "not provided"},
    {"city": "Mumbai", "use": "Commercial", "plot_area": 1200, "road_width": 18, "floors": 9, "height": 32, "fsi": 3.0, "front_setback": "not provided"},
    {"city": "Hyderabad", "use": "Residential", "plot_area": 500, "road_width": 12, "floors": 5, "height": 18, "front_setback": 3, "parking_spaces": "not provided"},
]
model = load_model("fp16")
for plan in PLANS:
    print("=" * 100, "\nPLAN:", plan, "\n")
    print(report_flow(model, tok, plan))
    print("\nPre-check only. Confirm with the municipal authority or an architect.")
model = None; clear_gpu()

## Notes

- Embeddings, chunking, FAISS and the abstention constant are identical to the app. The deployed app uses Groq's hosted model plus RAG; the LoRA adapters saved here are not loaded there.
- Q&A pairs are written by an LLM, so some gold answers are imperfect; token F1 also punishes correct answers phrased differently. Read the metrics as relative comparisons.
- Unanswerable questions are hand-written and the retrieval threshold was not tuned on them.